In [ ]:
# ==========================================
# NLGCL Training on Kaggle (All-in-One)
# ==========================================

import os
import shutil
import pandas as pd
import numpy as np
import sys
import zipfile

# 1. System & Dependency Fixes
print(">>> fixing build environment and installing dependencies...")
try:
    # Fix import errors in pip/setuptools contexts
    !pip install -U pip setuptools wheel
    
    # Enforce NumPy < 2.0 FIRST to avoid build/runtime errors with RecBole/PyTorch
    !pip install "numpy<2.0"
    
    # Install main dependencies (removed pyg_lib, pinned colorama)
    !pip install -q recbole colorlog tensorboard "colorama>=0.4.6" pyyaml pandas hyperopt==0.2.5 --no-build-isolation
    
    # Install kaggle cli if not present (usually pre-installed in Kaggle kernels)
    !pip install -q kaggle
except Exception as e:
    print(f"Warning: Dependency install had issues: {e}. Continuing to see if environment works...")

# 2. Install PyTorch Geometric
import torch
TORCH_version = torch.__version__
CUDA_version = torch.version.cuda
print(f"Torch: {TORCH_version}, CUDA: {CUDA_version}")

# Install PyG
!pip install -q torch-geometric
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv

# 3. Clone Repository
if not os.path.exists("NLGCL"):
    print(">>> Cloning Repository...")
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL
else:
    if os.getcwd().split(os.sep)[-1] != "NLGCL":
        %cd NLGCL

# 4. Patch Missing Modules
print(">>> Patching Missing Modules...")
missing_modules = [
    ("lightgcn", "LightGCN"), ("hmlet", "HMLET"), ("ncl", "NCL"), ("ngcf", "NGCF"),
    ("sgl", "SGL"), ("lightgcl", "LightGCL"), ("simgcl", "SimGCL"), ("xsimgcl", "XSimGCL"),
    ("directau", "DirectAU"), ("ssl4rec", "SSL4REC"), ("dccf", "DCCF"), ("l2cl", "L2CL")
]
base_path = "recbole_gnn/model/general_recommender"
if os.path.exists(base_path):
    for module_name, class_name in missing_modules:
        file_path = os.path.join(base_path, f"{module_name}.py")
        if not os.path.exists(file_path):
            with open(file_path, 'w') as f:
                f.write(f"class {class_name}:\n    pass\n")

# 5. Dataset Preparation (Tenrec QB-Video)
print(">>> Preparing QB-Video Dataset...")
dataset_name = "QB-video"
data_dir = f"dataset/{dataset_name}"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

inter_file = os.path.join(data_dir, f"{dataset_name}.inter")

if not os.path.exists(inter_file):
    print("Downloading QB-Video dataset via Kaggle API (Mirror: mamineelhilali/tenrec-dataset)...")
    
    # Configure Kaggle API credentials if missing
    # In a real Kaggle notebook, these are often auto-configured or need 'kaggle.json' uploaded.
    # However, for public datasets, lightweight download might work if env vars are set.
    # If this fails, user needs to upload kaggle.json or Add Data via GUI.
    
    # Try to download using Kaggle CLI
    # We download specific file 'QB-video.csv' or the whole zip if file specific not supported well
    try:
        # Download specific file from the dataset 'mamineelhilali/tenrec-dataset'
        !kaggle datasets download -d mamineelhilali/tenrec-dataset -f QB-video.csv -p {data_dir} --unzip
    except:
        print("Kaggle CLI download failed (missing credentials?). Trying separate manual download logic...")
    
    # Check if download succeeded
    csv_path = os.path.join(data_dir, 'QB-video.csv')
    if not os.path.exists(csv_path):
        # Fallback: Try a different mirror or direct link if Kaggle API fails
        # But realistically, on Kaggle, users should use "Add Data".
        print("Error: Dataset download failed. Please ADD DATA 'mamineelhilali/tenrec-dataset' manually in Kaggle sidebar.")
        print("Or ensure 'kaggle.json' is present for API usage.")
        
        # Creating a placeholder/error if we can't proceed
        # raise FileNotFoundError("QB-video.csv not found.")
    
    if os.path.exists(csv_path):
        print(f"Found CSV: {csv_path}")
        print("Converting to RecBole .inter format...")
        try:
            df = pd.read_csv(csv_path)
            
            # Map columns
            # Tenrec QB-Video usually has: user_id, video_id, click, like, share, follow...
            user_col = next((c for c in df.columns if 'user' in c.lower()), None)
            item_col = next((c for c in df.columns if 'item' in c.lower() or 'video' in c.lower()), None)
            
            if user_col and item_col:
                new_df = pd.DataFrame()
                new_df['user_id:token'] = df[user_col]
                new_df['item_id:token'] = df[item_col]
                new_df = new_df.drop_duplicates()
                new_df.to_csv(inter_file, index=False, sep='\t')
                print(f"Saved {inter_file} with {len(new_df)} interactions.")
            else:
                print(f"Error: Could not identify user/item columns in {df.columns.tolist()}")

        except Exception as e:
            print(f"Conversion Error: {e}")

# 6. Run Training
print(">>> Starting Training...")
!python main.py --dataset QB-video --model NLGCL --n_layers=4 --reg_weight=1e-4 --cl_temp=0.2 --alpha=0.6 --cl_reg=5e-5